# 04 — Diseño de carga PostGIS para MT Montecarlo

Este notebook diseña el primer flujo hacia PostgreSQL/PostGIS con un alcance controlado: media tensión de Montecarlo.

Para esto, se diseña un piloto verificable basado en `TMP_SHAPEFILE`, `Objetos_Red`, `SET`, `seccionadores`, `MT_cables` y `dxf/Montecarlo_MT.dxf`.


## 1. Objetivo

El objetivo es diseñar una migración por etapas:

- cargar datos crudos sin perder información;
- depurar claves y textos CAD;
- reconstruir geometrías desde `Datos_Objeto`;
- validar tipos geométricos por capa;
- enriquecer con atributos SQL Server;
- medir cobertura, rechazos y métricas antes de publicar en `gis`.


## 2. Preparación de rutas y librerías

Ubicamos la raíz del proyecto y la carpeta `sqlserver/exports/`. Los TSV de esa carpeta son insumos de auditoría, no todavía el modelo final de PostGIS.


In [9]:
# pathlib permite resolver rutas sin depender de la carpeta exacta desde donde se abre Jupyter.
from pathlib import Path

# subprocess se usa solo para consultas de lectura contra Docker/SQL Server cuando esté disponible.
import subprocess

# os permite leer variables de entorno cargadas desde .env.
import os

# pandas permite perfilar TSV y construir matrices de decisión legibles.
import pandas as pd

# Buscamos la raíz del proyecto subiendo hasta encontrar archivos propios del repo.
# No dependemos del nombre de la carpeta para que funcione aunque el repo se renombre.
RAIZ = Path.cwd()
while RAIZ.parent != RAIZ:
    if (RAIZ / 'docker-compose.yml').exists() and (RAIZ / 'scripts' / 'gis_sqlserver.sh').exists():
        break
    RAIZ = RAIZ.parent

# Cargamos .env local sin pisar variables ya definidas por el entorno.
def cargar_env_local() -> None:
    env_path = RAIZ / '.env'
    if not env_path.exists():
        return
    for line in env_path.read_text(encoding='utf-8').splitlines():
        line = line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, value = line.split('=', 1)
        os.environ.setdefault(key.strip(), value.strip())

cargar_env_local()

# Definimos rutas de trabajo del piloto.
EXPORTS = RAIZ / 'sqlserver' / 'exports'
DXF_MT = RAIZ / 'dxf' / 'Montecarlo_MT.dxf'
print('Raíz del proyecto:', RAIZ)
print('Exports:', EXPORTS)
print('DXF MT disponible:', DXF_MT.exists())


Raíz del proyecto: /home/martin/Code/gis_ceml
Exports: /home/martin/Code/gis_ceml/sqlserver/exports
DXF MT disponible: True


## 3. Confirmación de archivos TSV

Primero confirmamos qué tablas están exportadas. Si faltan, el diseño sigue siendo válido, pero la validación empírica queda pendiente.


In [10]:
# Listamos únicamente archivos TSV porque ese fue el formato elegido para preservar textos con comas.
archivos_tsv = sorted(EXPORTS.glob('*.tsv'))

# Marcamos las cinco tablas necesarias para la primera etapa MT.
tablas_piloto = ['tmp_shapefile', 'objetos_red', 'set', 'seccionadores', 'mt_cables']
existentes = {path.stem.lower() for path in archivos_tsv}
for tabla in tablas_piloto:
    estado = 'disponible' if tabla in existentes else 'pendiente'
    print(f'{tabla}: {estado}')


tmp_shapefile: disponible
objetos_red: disponible
set: disponible
seccionadores: disponible
mt_cables: disponible


## 4. Perfilado mínimo de TSV

Cargamos los TSV disponibles como texto para preservar claves, códigos CAD y valores con ceros a la izquierda. Este perfilado sirve para decidir qué entra a `crudo` y qué falta auditar.


In [11]:
# Cargamos TSV disponibles como texto para evitar conversiones automáticas peligrosas.
# Si una tabla todavía no existe, simplemente no aparece en el diccionario.
tablas = {}
for path in archivos_tsv:
    try:
        tablas[path.stem.lower()] = pd.read_csv(path, sep='	', dtype=str, keep_default_na=False)
    except Exception as exc:
        print(f'No se pudo leer {path.name}: {exc}')

# Perfilamos solo las tablas del piloto para mantener el alcance acotado.
perfil = []
for nombre in tablas_piloto:
    df = tablas.get(nombre)
    if df is None:
        perfil.append({'tabla': nombre, 'estado': 'pendiente', 'filas': 0, 'columnas': 0, 'columnas_criticas': ''})
        continue
    columnas_criticas = [c for c in df.columns if c.lower() in {'idcad', 'coop', 'datos_objeto'}]
    perfil.append({'tabla': nombre, 'estado': 'disponible', 'filas': len(df), 'columnas': len(df.columns), 'columnas_criticas': ', '.join(columnas_criticas)})

pd.DataFrame(perfil)


,tabla,estado,filas,columnas,columnas_criticas
0,tmp_shapefile,disponible,125688,6,"COOP, IDCAD, Datos_Objeto"
1,objetos_red,disponible,69134,18,"idcad, coop"
2,set,disponible,800,17,"idcad, coop"
3,seccionadores,disponible,1063,21,"coop, idcad"
4,mt_cables,disponible,40,3,


## 5. Esquemas propuestos en PostGIS

La propuesta usa lo siguiente:

| Esquema | Rol | Regla de uso |
|---|---|---|
| `crudo` | datos tal como llegan | no corregir ni perder columnas |
| `depuracion` | normalización y reconstrucción | claves limpias, parsing de `Datos_Objeto`, geometrías intermedias |
| `gis` | capas publicables | geometrías válidas y listas para QGIS |
| `auditoria` | trazabilidad | métricas, rechazos, cobertura y controles |


## 6. Diseño de flujo para MT

Secuencia propuesta:

1. `crudo.tmp_shapefile`, `crudo.objetos_red`, `crudo.set`, `crudo.seccionadores`, `crudo.mt_cables`: carga textual completa.
2. `depuracion.tmp_shapefile_parseado`: separar tokens `§`, clasificar entidad CAD y extraer coordenadas.
3. `depuracion.mt_geometrias`: construir geometría según entidad CAD.
4. `depuracion.mt_atributos`: normalizar `idcad/IDCAD` y `coop/COOP` para unir `Objetos_Red`.
5. `auditoria.mt_metricas_carga`: registrar cobertura, faltantes, geometrías inválidas y rechazos.
6. `gis.mt_cables`, `gis.mt_postes`, `gis.mt_elementos`, `gis.mt_seccionadores`, `gis.set`: publicar capas validadas.


In [17]:
# Esta matriz resume el diseño de etapas antes de escribir SQL real.
# La columna control indica qué se debe medir o validar en cada etapa.
flujo = pd.DataFrame([
    {'etapa': 'crudo', 'entrada': 'TSV MT', 'salida': 'tablas crudas', 'control': 'conteo de filas y columnas'},
    {'etapa': 'depuracion', 'entrada': 'Datos_Objeto', 'salida': 'entidad CAD clasificada', 'control': 'INSERT/CIRCLE/LINE/LWPOLYLINE detectados'},
    {'etapa': 'depuracion', 'entrada': 'tokens CAD', 'salida': 'geometría candidata', 'control': 'tipo geométrico esperado'},
    {'etapa': 'depuracion', 'entrada': 'Objetos_Red + TMP_SHAPEFILE', 'salida': 'atributos unidos', 'control': 'cobertura por idcad/coop'},
    {'etapa': 'auditoria', 'entrada': 'rechazos y métricas', 'salida': 'auditoria.mt_metricas_carga', 'control': 'trazabilidad de errores'},
    {'etapa': 'gis', 'entrada': 'geometrías válidas', 'salida': 'capas MT publicables', 'control': 'validación PostGIS'},
])
flujo


,etapa,entrada,salida,control
0,crudo,TSV MT,tablas crudas,conteo de filas y columnas
1,depuracion,Datos_Objeto,entidad CAD clasificada,INSERT/CIRCLE/LINE/LWPOLYLINE detectados
2,depuracion,tokens CAD,geometría candidata,tipo geométrico esperado
3,depuracion,Objetos_Red + TMP_SHAPEFILE,atributos unidos,cobertura por idcad/coop
4,auditoria,rechazos y métricas,auditoria.mt_metricas_carga,trazabilidad de errores
5,gis,geometrías válidas,capas MT publicables,validación PostGIS


## 7. Auditoría exhaustiva contra SQL Server

Antes de implementar la carga final conviene auditar la base restaurada buscando explícitamente las tablas y columnas:

- Tablas: `TMP_SHAPEFILE`, `Objetos_Red`, `SET`, `seccionadores`, `MT_cables`.
- Columnas: `IDCAD/idcad`, `COOP/coop`, `Datos_Objeto`.

La consulta es de solo lectura y puede ejecutarse si SQL Server está levantado.


In [13]:
# Esta función ejecuta comandos de solo lectura y devuelve el resultado sin interrumpir todo el notebook.
def ejecutar_comando(comando: list[str], timeout: int = 60) -> subprocess.CompletedProcess:
    # capture_output=True permite mostrar stdout/stderr de manera controlada en clase.
    return subprocess.run(comando, cwd=RAIZ, text=True, capture_output=True, timeout=timeout)

# Consulta de metadata enfocada en las tablas y columnas del piloto MT.
# No modifica datos: solo lee INFORMATION_SCHEMA.
consulta_auditoria = """
SELECT
    c.TABLE_SCHEMA,
    c.TABLE_NAME,
    c.COLUMN_NAME,
    c.DATA_TYPE,
    c.CHARACTER_MAXIMUM_LENGTH
FROM INFORMATION_SCHEMA.COLUMNS c
WHERE c.TABLE_NAME IN ('TMP_SHAPEFILE', 'Objetos_Red', 'SET', 'seccionadores', 'MT_cables')
   OR c.COLUMN_NAME IN ('IDCAD', 'idcad', 'COOP', 'coop', 'Datos_Objeto')
ORDER BY c.TABLE_NAME, c.ORDINAL_POSITION;
"""

# Ejecutamos la auditoría solo si Docker y el contenedor SQL Server están disponibles.
# Si no están activos, mostramos la consulta para ejecutarla después sin frenar la clase.
consulta_sqlcmd = consulta_auditoria.replace("\n", " ").replace('"', r'\"')
sa_password = os.getenv('MSSQL_SA_PASSWORD', 'CEML_Admin_2026!')
sqlcmd = f"SQLCMD=$([ -x /opt/mssql-tools18/bin/sqlcmd ] && echo /opt/mssql-tools18/bin/sqlcmd || echo /opt/mssql-tools/bin/sqlcmd); SQLCMD_TLS=$([ \"$SQLCMD\" = /opt/mssql-tools18/bin/sqlcmd ] && echo -C || echo ''); \"$SQLCMD\" $SQLCMD_TLS -S localhost -U SA -P '{sa_password}' -d CEML_GIS -Q \"{consulta_sqlcmd}\" -W -s $'\t'"
comando_auditoria = [
    'docker', 'compose', '-f', str(RAIZ / 'docker-compose.yml'),
    'exec', '-T', 'sqlserver', '/bin/bash', '-lc', sqlcmd,
]

try:
    resultado = ejecutar_comando(comando_auditoria, timeout=60)
    if resultado.returncode == 0:
        print(resultado.stdout)
    else:
        print('No se pudo ejecutar la auditoría contra SQL Server en este momento.')
        print('Consulta SQL para ejecutar cuando el contenedor esté activo:')
        print(consulta_auditoria)
        if resultado.stderr.strip():
            print('Detalle técnico:', resultado.stderr.strip())
except Exception as exc:
    print('SQL Server no está disponible para auditoría automática en este momento.')
    print('Consulta SQL para ejecutar cuando el contenedor esté activo:')
    print(consulta_auditoria)
    print('Detalle técnico:', exc)


TABLE_SCHEMA	TABLE_NAME	COLUMN_NAME	DATA_TYPE	CHARACTER_MAXIMUM_LENGTH
------------	----------	-----------	---------	------------------------
dbo	__RCL_RECLAMOS	COOP	varchar	5
dbo	__RCL_RECLAMOS	IDCAD	varchar	50
dbo	_ERR_SET	idcad	nvarchar	50
dbo	_ERR_SET	coop	nvarchar	5
dbo	ABONADOS_TEL	COOP	varchar	50
dbo	ABONADOS_TEL	IDCAD	varchar	50
dbo	alturas	coop	nvarchar	50
dbo	alturas	idcad	nvarchar	50
dbo	AP_ALUMBRADO	coop	nvarchar	5
dbo	AP_ALUMBRADO	idcad	nvarchar	50
dbo	ca_calles	coop	nvarchar	50
dbo	ca_calles	idCad	nvarchar	50
dbo	CA_Otros	coop	nvarchar	5
dbo	CA_Otros	idcad	nvarchar	50
dbo	cad_auxiliar	idcad	nvarchar	50
dbo	cad_auxiliar	coop	nvarchar	50
dbo	CalidadTecnica	coop	nvarchar	5
dbo	CalidadTecnica	idcad	nvarchar	50
dbo	Capas	Coop	varchar	255
dbo	CapasImg	Coop	varchar	255
dbo	catastro	coop	nvarchar	5
dbo	catastro	idCad	nvarchar	50
dbo	especies	coop	nvarchar	50
dbo	especies	idCad	nvarchar	50
dbo	GBL_Ordenes_Trabajo_Descripcion	coop	nvarchar	5
dbo	GBL_Ordenes_Trabajo_Descripcion	idca

## 8. Reconstrucción geométrica antes de publicar en `GIS`

La etapa de reconstrucción debe ocurrir antes de crear capas finales:

| Entidad CAD en `Datos_Objeto` | Geometría esperada | Validación |
|---|---|---|
| `INSERT` | `POINT` | coordenada de inserción presente |
| `CIRCLE` | `POINT` | centro presente; radio queda como atributo si hace falta |
| `LINE` | `LINESTRING` | dos puntos mínimos |
| `LWPOLYLINE` abierta | `LINESTRING` | dos vértices mínimos |
| `LWPOLYLINE` cerrada | `POLYGON` | anillo cerrado y válido |

Los rechazos deben guardarse en `auditoria`, no descartarse silenciosamente.


In [14]:
# Definimos una matriz de reglas de reconstrucción para convertirla luego en SQL o Python ETL.
# Esta matriz explicita qué tipo geométrico se acepta para cada entidad CAD.
reglas_reconstruccion = pd.DataFrame([
    {'entidad_cad': 'INSERT', 'geometria': 'POINT', 'criterio': 'usar punto de inserción'},
    {'entidad_cad': 'CIRCLE', 'geometria': 'POINT', 'criterio': 'usar centro del círculo'},
    {'entidad_cad': 'LINE', 'geometria': 'LINESTRING', 'criterio': 'usar punto inicial y final'},
    {'entidad_cad': 'LWPOLYLINE abierta', 'geometria': 'LINESTRING', 'criterio': 'usar secuencia de vértices'},
    {'entidad_cad': 'LWPOLYLINE cerrada', 'geometria': 'POLYGON', 'criterio': 'usar anillo cerrado y validar polígono'},
])
reglas_reconstruccion


,entidad_cad,geometria,criterio
0,INSERT,POINT,usar punto de inserción
1,CIRCLE,POINT,usar centro del círculo
2,LINE,LINESTRING,usar punto inicial y final
3,LWPOLYLINE abierta,LINESTRING,usar secuencia de vértices
4,LWPOLYLINE cerrada,POLYGON,usar anillo cerrado y validar polígono


## 9. Matriz de decisión inicial

| Tema | Decisión para la primera etapa | Justificación |
|---|---|---|
| `TMP_SHAPEFILE` | cargar en `crudo` y parsear en `depuracion` | contiene `Datos_Objeto` con geometría reconstruible |
| `Objetos_Red` | cargar y unir por `idcad/coop` | aporta atributos de red; cobertura esperada aproximada 68,66% |
| `SET` | enriquecer `gis.set` | punto MT |
| `seccionadores` | enriquecer `gis.mt_seccionadores` | punto MT |
| `MT_cables` | usar como soporte/catálogo | no asumir traza geométrica por sí solo |
| `Trafos` | fuera de geometría inicial | no tiene geometría/coordenadas/idcad directo; ubicación derivada de SET queda fuera del alcance actual |
| BT/suministros/alumbrado/reclamos | fases posteriores | dominios distintos y fuente CAD separada |
| Catastro completo | fase posterior | requiere etiquetas y contención espacial; no join directo por idcad en polígonos |
| `gis.ejes_viales` | fase posterior | usar este nombre si se incorporan capas viales/rurales |


In [ ]:
# Definimos las capas PostGIS objetivo de MT.
# Estas son las únicas capas gis propuestas para la primera publicación controlada.
capas_objetivo = pd.DataFrame([
    {'capa': 'gis.mt_cables', 'geometria': 'LINESTRING', 'fuente': '.00.MT_Cables'},
    {'capa': 'gis.mt_postes', 'geometria': 'POINT', 'fuente': '.00.MT_Postes'},
    {'capa': 'gis.mt_elementos', 'geometria': 'POINT', 'fuente': '.00.MT_Elementos'},
    {'capa': 'gis.mt_seccionadores', 'geometria': 'POINT', 'fuente': '.00.MT_Seccionadores'},
    {'capa': 'gis.set', 'geometria': 'POINT', 'fuente': '.00.SET'},
])
capas_objetivo


,capa,geometria,fuente
0,gis.mt_cables,LINESTRING,.00.MT_Cables
1,gis.mt_postes,POINT,.00.MT_Postes
2,gis.mt_elementos,POINT,.00.MT_Elementos
3,gis.mt_seccionadores,POINT,.00.MT_Seccionadores
4,gis.set,POINT,.00.SET


## 10. Métricas mínimas de auditoría

Registrar en `auditoria`:

- cantidad de filas cargadas por tabla cruda;
- cantidad de entidades CAD por tipo;
- cantidad de geometrías construidas por tipo;
- cantidad de geometrías inválidas o sin coordenadas suficientes;
- cobertura de unión `Objetos_Red` ↔ `TMP_SHAPEFILE` por `idcad/coop`;
- cantidad de entidades publicadas por capa `gis`;
- rechazos con causa reproducible.


In [16]:
# Esta tabla sirve como checklist de métricas para implementar después en SQL o ETL.
# Cada métrica debe poder recalcularse para auditar la carga.
metricas = pd.DataFrame([
    {'metrica': 'filas_crudo_por_tabla', 'destino': 'auditoria.mt_metricas_carga'},
    {'metrica': 'entidades_cad_por_tipo', 'destino': 'auditoria.mt_metricas_carga'},
    {'metrica': 'geometrias_construidas_por_tipo', 'destino': 'auditoria.mt_metricas_carga'},
    {'metrica': 'geometrias_invalidas_o_incompletas', 'destino': 'auditoria.mt_rechazos'},
    {'metrica': 'cobertura_objetos_red_tmp_shapefile', 'destino': 'auditoria.mt_metricas_carga'},
    {'metrica': 'publicados_por_capa_gis', 'destino': 'auditoria.mt_metricas_carga'},
])
metricas


,metrica,destino
0,filas_crudo_por_tabla,auditoria.mt_metricas_carga
1,entidades_cad_por_tipo,auditoria.mt_metricas_carga
2,geometrias_construidas_por_tipo,auditoria.mt_metricas_carga
3,geometrias_invalidas_o_incompletas,auditoria.mt_rechazos
4,cobertura_objetos_red_tmp_shapefile,auditoria.mt_metricas_carga
5,publicados_por_capa_gis,auditoria.mt_metricas_carga


## 11. Cierre del diseño de carga PostGIS

En esta notebook se definió el flujo de carga PostGIS de MT: esquemas de trabajo, tablas crudas, reglas de reconstrucción, capas finales y métricas mínimas de auditoría.

- El diseño conserva trazabilidad entre `crudo`, `depuracion`, `gis` y `auditoria`.
- La notebook 05 implementa la base PostGIS y la carga cruda de los cinco TSV.
